# PSF, SED and source morphology — complexity ladder (miniature)

This notebook walks through the **complexity system** introduced in `spectangle`:
every physical ingredient (PSF, SED, source morphology) exists at three levels
of realism, selectable with a single `complexity` integer.

| Complexity | PSF | SED | Source morphology |
|:---:|---|---|---|
| **0** | Gaussian | Gaussian spectral line | 2-D Gaussian |
| **1** | Moffat | Planck blackbody | Sérsic profile |
| **2** | Euclid NISP (Moffat + ellipticity + λ-scaling) | Realistic stellar atmosphere | Delta / disc / ring |

**Kernel:** use the `Python (spectangle)` kernel so that the package is importable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# PSF models
from spectangle.physics.psf import PSFModel, MoffatPSF, EuclidNISPPSF, make_psf

# SED models
from spectangle.simulations.sed import GaussianSED, BlackbodySED, RealisticSED, make_sed, random_sed

# Source morphology models
from spectangle.simulations.sources import (
    GaussianSource, SersicSource, PointSource, DiscGalaxy, RingNebula, make_source
)

print('Imports OK')

---
## 1  SED complexity ladder

We compare three SEDs at identical parameters across the NISP wavelength range
(9 250 – 18 500 Å):

* **Complexity 0** — Gaussian centred at 13 500 Å (σ = 1 500 Å)
* **Complexity 1** — Planck blackbody at T = 6 000 K
* **Complexity 2** — RealisticSED (currently wraps BB; hook for stellar templates)

In [ ]:
wav = np.linspace(9250, 18500, 300)   # Å

sed0 = make_sed(0, peak_wavelength_AA=13500, sigma_AA=1500)
sed1 = make_sed(1, temperature_K=6000)
sed2 = make_sed(2, temperature_K=6000, log_g=4.5, metallicity=0.0)

fig, ax = plt.subplots(figsize=(9, 4))
for sed, label, color in [
    (sed0, f'Complexity 0 — {sed0!r}', 'tab:blue'),
    (sed1, f'Complexity 1 — {sed1!r}', 'tab:orange'),
    (sed2, f'Complexity 2 — {sed2!r}', 'tab:green'),
]:
    flux = sed(wav)
    ax.plot(wav, flux, label=label, color=color)

ax.set_xlabel('Wavelength [Å]')
ax.set_ylabel('Normalised flux (unit integral)')
ax.set_title('SED complexity ladder')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Sanity check: no NaN / negative values
for name, sed in [('C0', sed0), ('C1', sed1), ('C2', sed2)]:
    f = sed(wav)
    assert np.all(np.isfinite(f)), f"{name}: non-finite values detected!"
    assert np.all(f >= 0), f"{name}: negative values detected!"
    print(f"{name} ({sed.__class__.__name__}): min={f.min():.2e}, max={f.max():.2e}, integral≈{np.trapz(f,wav):.3f}")

### 1b  Random SEDs — reproducibility and distribution

`random_sed(rng, complexity)` draws random parameters from physically motivated
priors and returns an SED instance.  The plot below shows five random draws at
each complexity level.

In [ ]:
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for c, ax in enumerate(axes):
    for _ in range(5):
        sed = random_sed(rng, complexity=c)
        ax.plot(wav, sed(wav), alpha=0.7)
    ax.set_title(f'Complexity {c} — {sed.__class__.__name__}')
    ax.set_xlabel('Wavelength [Å]')
    ax.set_ylabel('Flux (norm.)')
    ax.grid(alpha=0.3)
plt.suptitle('5 random SEDs per complexity level', y=1.02)
plt.tight_layout()
plt.show()

---
## 2  PSF complexity ladder

All PSF models expose a `.kernel` property (2-D numpy array normalised to unit
sum) and an `.at_wavelength(lam)` method that returns a scaled copy.

We compare:
* **Complexity 0** — Gaussian PSF
* **Complexity 1** — Moffat PSF (β = 3.5)
* **Complexity 2** — Euclid NISP PSF (Moffat + ellipticity + λ-scaling)

In [ ]:
fwhm_ref = 1.6   # pixels at 13 500 Å

psf0 = make_psf(0, fwhm_pixels=fwhm_ref)
psf1 = make_psf(1, fwhm_pixels=fwhm_ref, beta=3.5)
psf2 = make_psf(2, fwhm_ref_pixels=fwhm_ref, beta=3.5, ellipticity=0.85, pa_deg=30.0)

psf_labels = [
    f'C0 — {psf0!r}',
    f'C1 — {psf1!r}',
    f'C2 — {psf2!r}',
]
psfs = [psf0, psf1, psf2]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i, (psf, label) in enumerate(zip(psfs, psf_labels)):
    k = psf.kernel
    # ---- image ----
    axes[0, i].imshow(k, origin='lower', cmap='inferno')
    axes[0, i].set_title(label, fontsize=8)
    axes[0, i].axis('off')
    # ---- radial profile ----
    ctr = np.array(k.shape) // 2
    y, x = np.indices(k.shape)
    r = np.sqrt((x - ctr[1])**2 + (y - ctr[0])**2).ravel()
    v = k.ravel()
    rmax = int(r.max()) + 1
    bins = np.arange(rmax)
    digit = np.digitize(r, bins)
    prof = np.array([v[digit == b].mean() if np.any(digit == b) else 0.0 for b in bins])
    axes[1, i].semilogy(bins, np.maximum(prof, 1e-12), '-o', ms=4)
    axes[1, i].set_xlabel('Radius [pix]')
    axes[1, i].set_ylabel('Kernel (log)')
    axes[1, i].grid(alpha=0.3)
plt.suptitle('PSF kernels and radial profiles (log scale) — complexity 0 / 1 / 2', y=1.01)
plt.tight_layout()
plt.show()

### 2b  Wavelength-dependent PSF (diffraction-limited scaling)

For all complexities, `psf.at_wavelength(lam)` returns a new instance whose
FWHM is scaled by λ / λ_ref — reproducing diffraction-limited behaviour.  The
Euclid NISP PSF additionally supports a tunable power-law exponent via `power`.

In [ ]:
wavelengths_test = np.array([9500, 11500, 13500, 15500, 17500], dtype=float)
lam_ref = 13500.0

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['tab:blue', 'tab:orange', 'tab:green']
for psf, color, label in zip(psfs, colors, ['C0 Gaussian', 'C1 Moffat', 'C2 EuclidNISP']):
    fwhms_lam = [psf.at_wavelength(lam, ref_wavelength_AA=lam_ref).fwhm_pixels
                 for lam in wavelengths_test]
    ax.plot(wavelengths_test, fwhms_lam, '-o', color=color, label=label)

# Theoretical λ/D reference line
ax.plot(wavelengths_test, fwhm_ref * wavelengths_test / lam_ref,
        'k--', label='λ/D (theory)', lw=1.5)

ax.set_xlabel('Wavelength [Å]')
ax.set_ylabel('PSF FWHM [pixels]')
ax.set_title('PSF FWHM vs wavelength (diffraction-limited scaling)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3  Source morphology complexity ladder

We render three source types — point source, galaxy, and planetary nebula — at
each of the three complexity levels on a 64 × 64 pixel stamp, then convolve with
the complexity-1 Moffat PSF to show the final observable morphology.

| | Complexity 0 | Complexity 1 | Complexity 2 |
|---|---|---|---|
| **Point** | Gaussian (σ=0.5px) | Sérsic n=0.5 | Delta function |
| **Galaxy** | Gaussian (σ=3px) | Sérsic n=2, r_e=5px | Exponential disc + bulge |
| **Nebula** | Gaussian (σ=4px) | Sérsic n=1, ring-like | Emission ring + WD |


In [ ]:
ny, nx = 64, 64
xc, yc = nx / 2.0, ny / 2.0

source_configs = [
    # (label, type, complexity, kwargs)
    ('Point C0', 'point',  0, dict(sigma_pixels=0.5)),
    ('Point C1', 'point',  1, {}),
    ('Point C2', 'point',  2, {}),
    ('Galaxy C0', 'galaxy', 0, dict(sigma_pixels=3.0)),
    ('Galaxy C1', 'galaxy', 1, dict(r_e_pixels=5.0, sersic_n=2.0, ellipticity=0.75, pa_deg=45.0)),
    ('Galaxy C2', 'galaxy', 2, dict(h_r_pixels=5.0, r_b_pixels=2.0, bulge_fraction=0.25,
                                    ellipticity=0.65, pa_deg=45.0)),
    ('Nebula C0', 'nebula', 0, dict(sigma_pixels=4.0)),
    ('Nebula C1', 'nebula', 1, dict(r_e_pixels=4.0, sersic_n=0.5, ellipticity=0.9, pa_deg=20.0)),
    ('Nebula C2', 'nebula', 2, dict(inner_r_pixels=4.0, outer_r_pixels=7.0,
                                    central_fraction=0.08, ellipticity=0.9, pa_deg=20.0)),
]

psf_conv = make_psf(1, fwhm_pixels=fwhm_ref, beta=3.5)  # Moffat PSF for convolution

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
axes = axes.reshape(3, 6)

for idx, (label, stype, comp, kw) in enumerate(source_configs):
    row = idx // 3
    col_pre  = (idx % 3) * 2
    col_post = col_pre + 1

    src = make_source(stype, comp, **kw)
    stamp = src.render(ny, nx, xc, yc)

    # Check for NaN/inf in the stamp
    assert np.all(np.isfinite(stamp)), f"{label}: stamp contains non-finite values!"

    convolved = psf_conv.convolve(stamp)

    axes[row, col_pre].imshow(stamp, origin='lower', cmap='magma',
                              vmin=0, vmax=stamp.max() or 1)
    axes[row, col_pre].set_title(f'{label}\n(intrinsic)', fontsize=8)
    axes[row, col_pre].axis('off')

    axes[row, col_post].imshow(convolved, origin='lower', cmap='magma',
                               vmin=0, vmax=convolved.max() or 1)
    axes[row, col_post].set_title(f'{label}\n(+ PSF C1)', fontsize=8)
    axes[row, col_post].axis('off')

col_labels = []
for c in range(3):
    col_labels.append(f'Complexity {c} (raw)')
    col_labels.append(f'Complexity {c} (PSF-conv.)')

plt.suptitle(
    'Source morphologies (intrinsic & PSF-convolved)\n'
    'Rows: Point source / Galaxy / Planetary nebula    '
    'Column pairs: Complexity 0 / 1 / 2',
    y=1.01
)
plt.tight_layout()
plt.show()

print('All stamps finite and non-negative: ✓')

---
## 4  Enclosed energy fraction vs aperture radius

The plot below quantifies how much flux falls within a circular aperture of
radius *r* pixels for each (source type, complexity) combination **after PSF
convolution**.  This is relevant for aperture photometry calibration.

In [ ]:
def enclosed_fraction(image, radii):
    """Return enclosed flux fraction at each radius (from image centre)."""
    yc_int, xc_int = image.shape[0] // 2, image.shape[1] // 2
    y, x = np.indices(image.shape)
    r2 = (x - xc_int) ** 2 + (y - yc_int) ** 2
    total = float(image.sum())
    if total <= 0:
        return np.zeros(len(radii))
    return np.array([float(image[r2 <= r**2].sum()) / total for r in radii])

radii = np.linspace(0, 15, 60)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
stype_names = ['Point source', 'Galaxy', 'Planetary nebula']

for row_idx, stype_name in enumerate(stype_names):
    ax = axes[row_idx]
    group = [cfg for cfg in source_configs if cfg[0].startswith(stype_name.split()[0])]
    for label, stype, comp, kw in group:
        src = make_source(stype, comp, **kw)
        stamp = src.render(ny, nx, xc, yc)
        convolved = psf_conv.convolve(stamp)
        ef = enclosed_fraction(convolved, radii)
        ax.plot(radii, ef, label=f'C{comp}', lw=2)
    ax.axhline(0.5, color='gray', lw=1, ls='--', label='50% EEF')
    ax.set_title(stype_name)
    ax.set_xlabel('Aperture radius [pixels]')
    ax.set_ylabel('Enclosed energy fraction')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Enclosed energy fraction (EEF) after Moffat PSF convolution')
plt.tight_layout()
plt.show()

---
## 5  Spectral cubes — combining source morphology × SED × PSF

We build a monochromatic cube for each complexity, combining an SED with a
source morphology.  Each cube has shape `(n_lambda, ny, nx)`.  This is the
input format expected by `ForwardModel`.

In [ ]:
n_lam = 32
wav_cube = np.linspace(9250, 18500, n_lam)

configs_cube = [
    dict(complexity=0,
         sed=make_sed(0, peak_wavelength_AA=13500, sigma_AA=1500),
         src=make_source('point', 0, sigma_pixels=0.8),
         psf=make_psf(0, fwhm_pixels=fwhm_ref)),
    dict(complexity=1,
         sed=make_sed(1, temperature_K=6000),
         src=make_source('galaxy', 1, r_e_pixels=4.0, sersic_n=1.5),
         psf=make_psf(1, fwhm_pixels=fwhm_ref, beta=3.5)),
    dict(complexity=2,
         sed=make_sed(1, temperature_K=4500),   # BB for now; swap for RealisticSED
         src=make_source('nebula', 2, inner_r_pixels=4.0, outer_r_pixels=7.0),
         psf=make_psf(2, fwhm_ref_pixels=fwhm_ref, beta=3.5, ellipticity=0.85, pa_deg=30.0)),
]

cubes = []
for cfg in configs_cube:
    cube = np.zeros((n_lam, ny, nx), dtype=np.float32)
    morphology = cfg['src'].render(ny, nx, xc, yc)   # (ny, nx)
    sed_flux = cfg['sed'](wav_cube)                  # (n_lam,)
    # outer product: each wavelength slice scales morphology by SED flux
    for i in range(n_lam):
        cube[i] = morphology * sed_flux[i]
    # Convolve with wavelength-dependent PSF
    cube_conv = np.zeros_like(cube)
    for i, lam in enumerate(wav_cube):
        psf_lam = cfg['psf'].at_wavelength(lam, ref_wavelength_AA=13500.0)
        cube_conv[i] = psf_lam.convolve(cube[i])
    cubes.append(cube_conv)

# ---- visualise: broadband image and spectral profile at centre ----
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
complexity_titles = ['Complexity 0\n(Gaussian SED + point source + Gaussian PSF)',
                     'Complexity 1\n(Blackbody SED + Sérsic galaxy + Moffat PSF)',
                     'Complexity 2\n(Blackbody SED + ring nebula + EuclidNISP PSF)']

for i, (cube, title) in enumerate(zip(cubes, complexity_titles)):
    broadband = cube.sum(axis=0)
    axes[0, i].imshow(broadband, origin='lower', cmap='inferno')
    axes[0, i].set_title(title, fontsize=9)
    axes[0, i].axis('off')

    yci, xci = int(round(yc)), int(round(xc))
    spec_at_centre = cube[:, yci, xci]
    axes[1, i].plot(wav_cube, spec_at_centre, color=f'C{i}')
    axes[1, i].set_xlabel('Wavelength [Å]')
    axes[1, i].set_ylabel('Flux at centre pixel')
    axes[1, i].set_title(f'Spectral profile at centre ({xci},{yci})', fontsize=9)
    axes[1, i].grid(alpha=0.3)

plt.suptitle('3-D spectral cubes: broadband image (top) and centre spectrum (bottom)', y=1.01)
plt.tight_layout()
plt.show()

for i, c in enumerate(cubes):
    assert np.all(np.isfinite(c)), f"Cube C{i} contains NaN/inf!"
    print(f'Cube C{i}: shape={c.shape}, min={c.min():.3e}, max={c.max():.3e}  ✓')

---
## Takeaways

### Complexity system
| Component | C0 (debug / fast) | C1 (ML training) | C2 (realistic sim.) |
|---|---|---|---|
| **SED** | Gaussian line | Blackbody | Stellar atmosphere template |
| **PSF** | Circular Gaussian | Moffat (power-law wings) | Moffat + ellipticity + λ-scaling |
| **Source** | 2-D Gaussian blob | Sérsic profile | Disc/bulge / delta / ring |

### Key physics
- **PSF FWHM scales with wavelength** (FWHM ∝ λ at fixed aperture): use
  `.at_wavelength()` inside the forward model.
- **Moffat wings** (C1+) scatter more flux to large radii than a Gaussian —
  relevant for confusion in dense fields.
- **Source compactness** governs how much spectral information is recoverable
  from a single-angle spectrogram vs the four K-sequence angles.

### Next steps
- **Notebook 03** (`03_dispersion_traces.ipynb`): feed these cubes into
  `ForwardModel` and visualise the four K-sequence dispersed spectrograms.
- **Notebook 01** (`01_generate_mini_dataset.ipynb`): run `SimpleSimulator` to
  generate a full HDF5 training dataset using C1 SEDs and PSFs.